In [ ]:
# Créer le répertoire results
Path(CONFIG['results_dir']).mkdir(parents=True, exist_ok=True)

# Sauvegarder les métriques
metrics = {
    'accuracy': float(accuracy_score(test_labels, test_preds)),
    'precision': float(precision_score(test_labels, test_preds, zero_division=0)),
    'recall': float(recall_score(test_labels, test_preds, zero_division=0)),
    'f1_score': float(f1_score(test_labels, test_preds, zero_division=0)),
    'auc': float(roc_auc_score(test_labels, test_probs[:, 1]))
}

with open(f"{CONFIG['results_dir']}/test_metrics.json", 'w') as f:
    json.dump(metrics, f, indent=4)

# Sauvegarder l'historique d'entraînement
training_history = {
    'train_losses': train_losses,
    'val_losses': val_losses,
    'val_accuracies': val_accuracies,
    'best_epoch': int(best_epoch)
}

with open(f"{CONFIG['results_dir']}/training_history.json", 'w') as f:
    json.dump(training_history, f, indent=4)

print("✅ Résultats sauvegardés:")
print(f"   Model: {CONFIG['model_save_path']}")
print(f"   Metrics: {CONFIG['results_dir']}/test_metrics.json")
print(f"   History: {CONFIG['results_dir']}/training_history.json")

## 10. Sauvegarder les résultats finaux

In [ ]:
# Courbe ROC
fpr, tpr, _ = roc_curve(test_labels, test_probs[:, 1])
auc = roc_auc_score(test_labels, test_probs[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Pneumonia Detection')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de confusion
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['NORMAL', 'PNEUMONIA'],
    yticklabels=['NORMAL', 'PNEUMONIA'],
    cbar_kws={'label': 'Count'}
)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix - Pneumonia Detection')
plt.tight_layout()
plt.show()

# Rapport de classification
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(test_labels, test_preds, target_names=['NORMAL', 'PNEUMONIA']))

## 9. Visualiser les résultats

In [ ]:
# Charger le meilleur modèle
model.load_state_dict(torch.load(CONFIG['model_save_path']))

# Évaluer sur test set
def evaluate(dataloader, verbose=True):
    """Évalue le modèle et retourne predictions, probabilités"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Testing"):
            images = batch['images'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # Calculer les métriques
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs[:, 1])
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"TEST SET RESULTS")
        print('='*60)
        print(f"Accuracy:  {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-Score:  {f1:.4f}")
        print(f"AUC-ROC:   {auc:.4f}")
        print('='*60)
    
    return all_preds, all_labels, all_probs

# Évaluer
test_preds, test_labels, test_probs = evaluate(test_loader, verbose=True)

## 8. Évaluer sur le test set

In [ ]:
# Afficher les courbes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(val_accuracies, label='Val Accuracy', linewidth=2, color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Visualiser les courbes d'entraînement

In [ ]:
def train_epoch(dataloader):
    """Entraîne pour une époque"""
    model.train()
    total_loss = 0.0
    
    for batch in tqdm(dataloader, desc="Training"):
        images = batch['images'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def validate_epoch(dataloader):
    """Valide le modèle"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating"):
            images = batch['images'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    return total_loss / len(dataloader), accuracy

# Boucle d'entraînement
for epoch in range(CONFIG['num_epochs']):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{CONFIG['num_epochs']}")
    print('='*60)
    
    # Entraîner
    train_loss = train_epoch(train_loader)
    train_losses.append(train_loss)
    
    # Valider
    val_loss, val_acc = validate_epoch(val_loader)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Learning rate scheduler
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        # Sauvegarder le meilleur modèle
        torch.save(model.state_dict(), CONFIG['model_save_path'])
        print(f"✅ Modèle sauvegardé (meilleur jusqu'à présent)")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['early_stopping_patience']:
            print(f"\n⏹️ Early stopping à l'époque {epoch+1}")
            break

print(f"\n✅ Entraînement terminé (meilleure époque: {best_epoch+1})")

In [ ]:
# Critère de perte avec class weights
class_weights = torch.tensor(CONFIG['class_weights'], dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer (seulement sur les paramètres entraînables)
optimizer = optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    verbose=True
)

# Variables pour le tracking
train_losses = []
val_losses = []
val_accuracies = []
best_val_loss = float('inf')
patience_counter = 0

print(f"✅ Entraînement configuration:")
print(f"   Loss: CrossEntropyLoss (class weighted)")
print(f"   Optimizer: Adam (lr={CONFIG['learning_rate']})")
print(f"   Scheduler: ReduceLROnPlateau (patience=3)")
print(f"   Early stopping patience: {CONFIG['early_stopping_patience']}")
print(f"\n🚀 Démarrage de l'entraînement...\n")

## 6. Entraîner le modèle

In [ ]:
# Créer ResNet50 avec couche de classification personnalisée
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Geler les couches pré-entraînées (transfer learning)
for param in model.parameters():
    param.requires_grad = False

# Remplacer la couche finale pour la classification binaire
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 2)  # 2 classes (normal, pneumonia)
)

# Dégeler les derniers blocs pour fine-tuning
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

# Placer le modèle sur le device
device = CONFIG['device']
model = model.to(device)

# Afficher info du modèle
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Modèle créé:")
print(f"   Architecture: ResNet50 (pré-entraîné ImageNet)")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Device: {device}")

## 5. Créer le modèle ResNet50

In [ ]:
# Transforms pour entraînement (avec augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD)
])

# Transforms pour test (sans augmentation)
test_transforms = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD)
])

def collate_fn_train(batch):
    """Prépare un batch pour l'entraînement"""
    images = []
    labels = []
    for item in batch:
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        images.append(train_transforms(img))
        labels.append(item['label'])
    return {
        'images': torch.stack(images),
        'labels': torch.tensor(labels, dtype=torch.long)
    }

def collate_fn_test(batch):
    """Prépare un batch pour le test"""
    images = []
    labels = []
    for item in batch:
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        images.append(test_transforms(img))
        labels.append(item['label'])
    return {
        'images': torch.stack(images),
        'labels': torch.tensor(labels, dtype=torch.long)
    }

# Créer train/val split
val_size = int(len(train_dataset) * CONFIG['val_split'])
train_size = len(train_dataset) - val_size

torch.manual_seed(CONFIG['random_seed'])
train_split, val_split = random_split(train_dataset, [train_size, val_size])

# Dataloaders
train_loader = DataLoader(
    train_split,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    collate_fn=collate_fn_train
)

val_loader = DataLoader(
    val_split,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    collate_fn=collate_fn_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    collate_fn=collate_fn_test
)

print(f"✅ Dataloaders créés:")
print(f"   Train: {len(train_loader)} batches")
print(f"   Val:   {len(val_loader)} batches")
print(f"   Test:  {len(test_loader)} batches")

## 4. Préparation des données & Dataloaders

In [ ]:
# Afficher des exemples
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
fig.suptitle('Exemples de Radiographies Thoraciques', fontsize=16, fontweight='bold')

# Classe 0 (Normal)
normal_indices = [i for i in range(len(full_dataset)) if full_dataset[i]['label'] == 0][:4]
for idx, i in enumerate(normal_indices):
    img = full_dataset[i]['image']
    axes[0, idx].imshow(img, cmap='gray')
    axes[0, idx].set_title(f'NORMAL', fontsize=11)
    axes[0, idx].axis('off')

# Classe 1 (Pneumonia)
pneumonia_indices = [i for i in range(len(full_dataset)) if full_dataset[i]['label'] == 1][:4]
for idx, i in enumerate(pneumonia_indices):
    img = full_dataset[i]['image']
    axes[1, idx].imshow(img, cmap='gray')
    axes[1, idx].set_title(f'PNEUMONIA', fontsize=11, color='red', fontweight='bold')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()

print("✅ Images visualisées")

## 3. Visualiser quelques images du dataset

In [ ]:
# Importer la fonction de chargement
from src.data_loader import load_raw_pneumonia_dataset

# Charger train et test
print("Chargement du dataset Kaggle...")
train_dataset = load_raw_pneumonia_dataset("train")
test_dataset = load_raw_pneumonia_dataset("test")

# Combiner
from datasets import concatenate_datasets
full_dataset = concatenate_datasets([train_dataset, test_dataset])

print(f"\n✅ Dataset chargé:")
print(f"   Train: {len(train_dataset)} images")
print(f"   Test:  {len(test_dataset)} images")
print(f"   Total: {len(full_dataset)} images")
print(f"   Classes: {full_dataset.column_names}")

# Afficher la distribution
labels = [full_dataset[i]['label'] for i in range(len(full_dataset))]
label_counts = Counter(labels)
print(f"\n📊 Distribution des classes:")
for cls, count in sorted(label_counts.items()):
    pct = 100 * count / len(full_dataset)
    print(f"   {CLASS_NAMES[cls]}: {count} images ({pct:.1f}%)")

## 2. Charger le Dataset

In [ ]:
# Configuration
CONFIG = {
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'img_size': 224,
    'batch_size': 32,
    'num_epochs': 15,
    'learning_rate': 0.0001,
    'weight_decay': 1e-4,
    'val_split': 0.2,
    'early_stopping_patience': 5,
    'random_seed': 42,
    
    # Class weights (pneumonia=4273, normal=1583 du dataset)
    'class_weights': [2.7, 1.0],
    
    # Paths
    'model_save_path': '../models/resnet50_pneumonia.pth',
    'results_dir': '../results/',
}

# Normalisation ImageNet
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

# Labels
CLASS_NAMES = {0: 'NORMAL', 1: 'PNEUMONIA'}

print("Configuration définie:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 1. Configuration & Hyperparamètres

In [ ]:
# Imports principaux
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import models, transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter
import json
import os
from pathlib import Path

# Dataset
from datasets import load_dataset, concatenate_datasets
import kagglehub

# Métriques
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

print("✅ Toutes les librairies importées avec succès")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ GPU disponible: {torch.cuda.is_available()}")

# Détection de Pneumonie - Pipeline d'Entraînement Complet

**Objectif**: Entraîner un modèle ResNet50 pour détecter la pneumonie sur des radiographies thoraciques.

**Dataset**: [Pneumonia Radiography Dataset - Kaggle](https://www.kaggle.com/datasets/iamtanmayshukla/pneumonia-radiography-dataset)

**Approche**: Transfer Learning (ResNet50 pré-entraîné) + Class Weights pour gérer le déséquilibre